In [1]:
import pertpy as pt
import scanpy as sc
from genetic_perturbation_playground.data.data_loader import load_dataset


adata = load_dataset("replogle_k562")

Loaded from /nfs/ghome/live/adhir/genetic_playground/data/replogle_k562_preprocessed.h5ad
  310,385 cells  ×  5,000 genes  |  2,058 perturbations


In [2]:
import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

target_gene = "RPL3"

ctrl_idx = np.where(adata.obs["perturbation"] == "control")[0]
rpl3_idx = np.where(adata.obs["perturbation"] == target_gene)[0]

ctrl_train_idx, ctrl_test_idx = train_test_split(ctrl_idx, test_size=0.2, random_state=42)
rpl3_train_idx, rpl3_test_idx = train_test_split(rpl3_idx, test_size=0.2, random_state=42)

def make_mask(indices):
    mask = np.zeros(adata.n_obs, dtype=bool)
    mask[indices] = True
    return mask

train_adata     = adata[make_mask(np.concatenate([ctrl_train_idx, rpl3_train_idx]))].copy()
ctrl_test_adata = adata[make_mask(ctrl_test_idx)].copy()
rpl3_test_adata = adata[make_mask(rpl3_test_idx)].copy()

train_adata.obs["condition"] = [
    "ctrl" if p == "control" else "stim"
    for p in train_adata.obs["perturbation"]
]
ctrl_test_adata.obs["condition"] = "ctrl"

print(f"Train: {len(ctrl_train_idx):,} ctrl + {len(rpl3_train_idx):,} RPL3")
print(f"Predict from: {len(ctrl_test_idx):,} held-out control cells")
print(f"Evaluate against: {len(rpl3_test_idx):,} held-out RPL3 cells")

Train: 8,552 ctrl + 1,596 RPL3
Predict from: 2,139 held-out control cells
Evaluate against: 400 held-out RPL3 cells


In [3]:
# Train scGen
pt.tl.Scgen.setup_anndata(train_adata, batch_key="condition")
model = pt.tl.Scgen(train_adata)
model.train(
    max_epochs=100,
    batch_size=32,
    early_stopping=True,
    early_stopping_patience=25,
    accelerator="gpu",
)

INFO     Jax module moved to cuda:0.Note: Pytorch lightning will show GPU is not being used for the Trainer.       


/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /nfs/ghome/live/adhir/genetic_playground/.venv/lib/ ...
/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /nfs/ghome/live/adhir/genetic_playground/.venv/lib/ ...
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='g

Training:   0%|          | 0/100 [00:00<?, ?it/s]

/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:378: You have overridden `transfer_batch_to_device` in `LightningModule` but have passed in a `LightningDataModule`. It will use the implementation from `LightningModule` instance.


Monitored metric elbo_validation did not improve in the last 25 records. Best score: 399.292. Signaling Trainer to stop.


In [4]:
pred, delta = model.predict(ctrl_key="ctrl", stim_key="stim", adata_to_predict=ctrl_test_adata)
pred.obs["condition"] = "pred"

def to_dense(X):
    return np.asarray(X.todense()) if sp.issparse(X) else np.asarray(X)

# scgen predicts the expression of the held out control cells
# we mean them and compare against the mean of the held out RPL3 cells (ground truth)
mean_pred   = to_dense(pred.X).mean(axis=0)
mean_actual = to_dense(rpl3_test_adata.X).mean(axis=0)
mean_ctrl   = to_dense(ctrl_test_adata.X).mean(axis=0)

# We subtract the control mean to focus on the perturbation effect, and compute Pearson correlation as a measure of similarity
r_pred, _ = pearsonr(mean_pred - mean_ctrl, mean_actual - mean_ctrl)

print(f"Evaluation — {target_gene}")
print(f"  R² predicted vs actual : {r_pred**2:.4f}")


INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/nfs/ghome/live/adhir/genetic_playground/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
Evaluation — RPL3
  R² predicted vs actual : 0.0987


In [5]:
pred.X

array([[0.02896821, 0.28345984, 0.0673669 , ..., 0.05706047, 0.04391637,
        0.10569201],
       [0.0113334 , 0.43132222, 0.04678747, ..., 0.0257866 , 0.02289204,
        0.0682198 ],
       [0.02987243, 0.41051227, 0.08360037, ..., 0.04034447, 0.03349227,
        0.04227263],
       ...,
       [0.02276531, 0.45409796, 0.04751123, ..., 0.01523805, 0.02494866,
        0.05412494],
       [0.02334454, 0.45965862, 0.0364138 , ..., 0.0314381 , 0.04714802,
        0.06293502],
       [0.02142033, 0.42126524, 0.03139674, ..., 0.04332275, 0.04904412,
        0.10837305]], shape=(2139, 5000), dtype=float32)